In [ ]:
#Step 1: Load and clean data
import pandas as pd

df = pd.read_csv('../data/lyrics_small_dataset.csv')

# Critic validation: lyrics and id are required
df = df[df['lyrics'].notna()]  # Remove NaN
df = df[df['lyrics'].str.strip() != '']  # Remove empty strings
df = df[df['id'].notna()]  # ID is required

# Important metadata validation
df = df[df['title'].notna()] 
df = df[df['artist'].notna()]

# Limpiar espacios en blanco en strings
df['lyrics'] = df['lyrics'].str.strip()
df['title'] = df['title'].str.strip()
df['artist'] = df['artist'].str.strip()

# Verify ID duplicates (if necessary)
print(f"ID duplicates: {df['id'].duplicated().sum()}")

# Get clean data
data = df.sample(1000).to_dict('records')
len(data)


In [ ]:
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

In [ ]:
# https://huggingface.co/sentence-transformers/models

encoder = SentenceTransformer('all-mpnet-base-v2') # Model to create embeddings

#Alternative models:
# 'all-MiniLM-L6-v2' (Lightning fast 90MB aprox)
# 'all-mpnet-base-v2' (balanced performance/size 438MB aprox)
# 'BAAI/bge-large-en-v1.5' (high precision 1.3GB aprox)
# 'BAAI/bge-m3' (multilingual 4.5GB aprox)

In [ ]:
# create the vector database client
qdrant = QdrantClient(path="../data/qdrant_db") # Create local storage in file
#qdrant = QdrantClient(":memory:") # Create in-memory Qdrant instance


In [ ]:
# Create collection to store songs
collection_name="top_songs"

if qdrant.collection_exists(collection_name):
    qdrant.delete_collection(collection_name) # Delete collection if exists

qdrant.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=encoder.get_sentence_embedding_dimension(), # Vector size is defined by used model
        distance=models.Distance.COSINE
    )
)

In [ ]:
# vectorize!
qdrant.upload_points(
    collection_name=collection_name,
    points=[
        models.PointStruct(
            id=idx,
            vector=encoder.encode(doc["lyrics"]).tolist(),
            payload=doc,
        ) for idx, doc in enumerate(data) # data is the variable holding all the songs
    ]
)


In [ ]:
# Search time for awesome songs!
user_prompt = "Suggest me a good song about love for money"

hits = qdrant.query_points(
    collection_name=collection_name,
    query=encoder.encode(user_prompt).tolist(),
    limit=5,
    with_payload=True,
    with_vectors=False,
)

for p in hits.points:
    print(p.payload, "score:", p.score)

# define a variable to hold the search results
search_results = [hit.payload for hit in hits.points]

In [ ]:
# Now it's time to connect to the local large language model
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",  # Local Ollama
    api_key="ollama"  # Ollama no requiere key real
)

#Roles:
#system: Defines the assistant's behavior and context (once at the beginning).
#user: User messages (questions, instructions).
#assistant: Previous assistant responses (to maintain history or provide examples).
completion = client.chat.completions.create(
    model="gpt-oss:20b",
    messages=[
        {
            "role": "system",
            "content": """You are a music recommendation assistant. You MUST ONLY recommend songs from the list provided by the user. 
            Do NOT invent or suggest songs that are not in the provided list. 
            Analyze the songs provided and explain why they match the user's request based on their lyrics and metadata."""
        },
        {
            "role": "user",
            "content": f"""I found these songs in my database that might match your request: {search_results}
            User request: {user_prompt}
            IMPORTANT: You must ONLY recommend songs from the list above. Do not suggest any other songs. Analyze each song and explain which one(s) best match the request and why."""
        }
    ]
)


In [ ]:
# Print LLM response

# Whole response
#print(completion.choices[0].message)

#Text response only
#response = completion.choices[0].message.content
#print(response)

#Markdown response
from IPython.display import Markdown
response = completion.choices[0].message.content
display(Markdown(response))

